## Results Summary and Hyperparameter Tuning

### Results

| Model | F1 (weighted) | Accuracy | Notes |
|---|---|---|---|
| DistilBERT u6 (all 6 layers) | 0.7915 | 0.7913 | **Best overall** |
| DistilBERT u4 (top 4 layers) | 0.7878 | 0.7875 | Close second |
| DistilBERT u2 (top 2 layers) | 0.7653 | 0.7653 | Good tradeoff |
| DistilBERT u0 (frozen)       | 0.5940 | 0.5966 | Too little fine-tuning |
| XGBoost                      | 0.6972 | 0.6976 | Best classical model |
| Random Forest                | 0.6784 | 0.6801 | Decent baseline |
| LSTM                         | 0.2330 | 0.4045 | Collapsed to neutral (class imbalance issue) |

---

### Key Findings

**Best Model: DistilBERT u6**
- Unfreezing all 6 transformer layers gave the best F1 of **0.7915** and accuracy of **0.7913**
- More trainable parameters (64.4% of total) = better feature learning for this dataset
- Positive class had the best per-class F1 (0.83), negative was slightly weaker (0.80)

**Effect of unfreezing layers (DistilBERT)**
- u0 (frozen): 0.5940 — the pretrained features alone are not enough
- u2: 0.7653 — big jump, top 2 layers already capture a lot
- u4: 0.7878 — further improvement, diminishing returns start here
- u6: 0.7915 — marginal gain over u4, but still the best

**Classical models (TF-IDF based)**
- XGBoost outperformed Random Forest (0.697 vs 0.678)
- Both used improved TF-IDF settings: max_features=50000, ngram_range=(1,3), sublinear_tf=True
- The ceiling for TF-IDF based models on this task appears to be around 0.70

**LSTM — collapsed to predicting neutral**
- Suffered from class imbalance: neutral (11117) >> negative (7781) >> positive (8582)
- Model defaulted to always predicting neutral, resulting in near-zero F1 for negative and positive
- Class weights were applied but the Bidirectional fix was not properly applied in this run
- LSTM results excluded from final comparison

---

### Hyperparameter Choices

**TF-IDF (for RF and XGBoost)**
- `max_features=50000` — wider vocabulary captures more tweet-specific terms
- `ngram_range=(1,3)` — captures unigrams, bigrams, trigrams (e.g. "not good at all")
- `sublinear_tf=True` — log-scales term frequency, reduces dominance of common words
- `min_df=2` — ignores terms appearing only once (noise reduction)

**XGBoost**
- `n_estimators=500` — more trees = better ensemble
- `learning_rate=0.1` — standard learning rate, balances speed and accuracy
- `max_depth=6` — controls tree complexity, prevents overfitting

**DistilBERT**
- `lr=2e-5` — standard fine-tuning learning rate for transformer models
- `epochs=3` — transformers converge fast, 3 epochs is typically sufficient
- `batch_size=32` — balances memory usage and gradient stability
- `max_len=64` — tweets are short, 64 tokens is sufficient coverage

In [1]:
!pip install xgboost lightgbm torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [2]:
df = pd.read_csv('../data/output.csv')
df = df[['cleaned_text', 'sentiment']].dropna()

le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])
print(df['sentiment'].value_counts())  # check class distribution

sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

In [4]:
tfidf = TfidfVectorizer(
    max_features=50000,   # was 10000, more features = more info
    ngram_range=(1, 3),   # was (1,2), captures more phrases
    sublinear_tf=True,    # dampens high frequency words
    min_df=2
)
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_tf, y_train)
rf_preds = rf.predict(X_test_tf)
print(classification_report(y_test, rf_preds, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.75      0.52      0.62      1556
     neutral       0.60      0.79      0.68      2223
    positive       0.78      0.68      0.73      1717

    accuracy                           0.68      5496
   macro avg       0.71      0.66      0.68      5496
weighted avg       0.70      0.68      0.68      5496



In [6]:
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train_tf, y_train)
xgb_preds = xgb.predict(X_test_tf)
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.76      0.57      0.65      1556
     neutral       0.62      0.78      0.69      2223
    positive       0.80      0.70      0.75      1717

    accuracy                           0.70      5496
   macro avg       0.72      0.69      0.70      5496
weighted avg       0.71      0.70      0.70      5496



In [7]:
results = {
    "Random Forest": {
        "Accuracy": accuracy_score(y_test, rf_preds),
        "F1 (weighted)": f1_score(y_test, rf_preds, average='weighted')
    },
    "XGBoost": {
        "Accuracy": accuracy_score(y_test, xgb_preds),
        "F1 (weighted)": f1_score(y_test, xgb_preds, average='weighted')
    },
}
print(pd.DataFrame(results).T)

               Accuracy  F1 (weighted)
Random Forest  0.680131       0.678353
XGBoost        0.697598       0.697158


In [8]:
!pip install tensorflow

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [9]:
MAX_WORDS = 20000   # vocabulary size
MAX_LEN = 50        # max tweet length in words

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print("Train shape:", X_train_pad.shape)
print("Test shape: ", X_test_pad.shape)

Train shape: (21984, 50)
Test shape:  (5496, 50)


In [10]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Class weights: {0: np.float64(1.1771887550200804), 1: np.float64(0.8239262424106139), 2: np.float64(1.067443554260743)}


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,        # reduced from 64
    validation_split=0.1,
    class_weight=class_weight_dict,   # this is the key fix
    callbacks=[early_stop]
)

Epoch 1/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 23s 33ms/step - accuracy: 0.3392 - loss: 1.0995 - val_accuracy: 0.2819 - val_loss: 1.1030
Epoch 2/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.3267 - loss: 1.0992 - val_accuracy: 0.4020 - val_loss: 1.0917
Epoch 3/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.3283 - loss: 1.0990 - val_accuracy: 0.3161 - val_loss: 1.1022
Epoch 4/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.3300 - loss: 1.0989 - val_accuracy: 0.2819 - val_loss: 1.0990
Epoch 5/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.3213 - loss: 1.0988 - val_accuracy: 0.2819 - val_loss: 1.1007


In [12]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

lstm_preds_prob = model.predict(X_test_pad)
lstm_preds = np.argmax(lstm_preds_prob, axis=1)

print(classification_report(y_test, lstm_preds, target_names=le.classes_))

172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00      1556
     neutral       0.40      1.00      0.58      2223
    positive       0.00      0.00      0.00      1717

    accuracy                           0.40      5496
   macro avg       0.13      0.33      0.19      5496
weighted avg       0.16      0.40      0.23      5496



/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [13]:
!pip install transformers datasets torch

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
bert_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts, labels, max_len=64):
    encodings = bert_tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors='pt'
    )
    return encodings, torch.tensor(list(labels), dtype=torch.long)

print("Tokenizing...")
train_encodings, train_labels_tensor = tokenize(X_train, y_train)
test_encodings,  test_labels_tensor  = tokenize(X_test,  y_test)
print("Done!")

Tokenizing...
Done!


In [15]:
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = TweetDataset(train_encodings, train_labels_tensor)
test_dataset  = TweetDataset(test_encodings,  test_labels_tensor)

print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")

Train size: 21984
Test size:  5496


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def build_distilbert(unfreeze_layers=0):
    """
    Load DistilBERT and selectively unfreeze transformer layers.
    DistilBERT has 6 transformer layers total.
    unfreeze_layers=0 → only classification head trains (frozen transformer)
    unfreeze_layers=6 → all layers fine-tune (full model)
    """
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=3
    )

    # Step 1: freeze everything
    for param in model.distilbert.parameters():
        param.requires_grad = False

    # Step 2: unfreeze the last n transformer layers
    if unfreeze_layers > 0:
        for layer in model.distilbert.transformer.layer[-unfreeze_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"u{unfreeze_layers} — Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    return model.to(device)

Using device: cpu


In [17]:
def train_model(model, train_dataset, num_epochs=3, batch_size=32):
    """Fine-tune a model and return it."""
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
    total_steps = num_epochs * len(train_loader)
    scheduler = get_scheduler("linear", optimizer=optimizer,
                               num_warmup_steps=0, num_training_steps=total_steps)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        correct = 0
        total   = 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

        print(f"  Epoch {epoch+1}/{num_epochs} — Loss: {total_loss/len(train_loader):.4f} — Acc: {correct/total:.4f}")

    return model

In [18]:
def evaluate_model(model, test_dataset, batch_size=32):
    """Run inference and return predictions."""
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

In [19]:
# Run experiments for u0, u2, u4, u6
# u0 = frozen,  u2 = top 2 layers,  u4 = top 4 layers,  u6 = all layers

experiment_results = {}

for n_layers in [0, 2, 4, 6]:
    run_name = f"distilbert_u{n_layers}"
    print(f"\n{'='*50}")
    print(f"Running: {run_name}")
    print(f"{'='*50}")

    # Build fresh model for each run
    model = build_distilbert(unfreeze_layers=n_layers)

    # Train
    model = train_model(model, train_dataset, num_epochs=3, batch_size=32)

    # Evaluate
    preds, labels = evaluate_model(model, test_dataset)

    f1  = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    experiment_results[run_name] = {"F1 (weighted)": round(f1, 4), "Accuracy": round(acc, 4)}

    print(f"\n{run_name} → F1: {f1:.4f} | Accuracy: {acc:.4f}")
    print(classification_report(labels, preds, target_names=le.classes_))


Running: distilbert_u0


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8974.46it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u0 — Trainable params: 592,899 / 66,955,779 (0.9%)
  Epoch 1/3 — Loss: 1.0273 — Acc: 0.4725
  Epoch 2/3 — Loss: 0.9368 — Acc: 0.5666
  Epoch 3/3 — Loss: 0.9044 — Acc: 0.5888

distilbert_u0 → F1: 0.5940 | Accuracy: 0.5966
              precision    recall  f1-score   support

    negative       0.67      0.46      0.54      1556
     neutral       0.53      0.71      0.61      2223
    positive       0.67      0.58      0.62      1717

    accuracy                           0.60      5496
   macro avg       0.62      0.58      0.59      5496
weighted avg       0.62      0.60      0.59      5496


Running: distilbert_u2


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15868.28it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u2 — Trainable params: 14,768,643 / 66,955,779 (22.1%)
  Epoch 1/3 — Loss: 0.6791 — Acc: 0.7031
  Epoch 2/3 — Loss: 0.5684 — Acc: 0.7623
  Epoch 3/3 — Loss: 0.5377 — Acc: 0.7794

distilbert_u2 → F1: 0.7653 | Accuracy: 0.7653
              precision    recall  f1-score   support

    negative       0.76      0.77      0.77      1556
     neutral       0.73      0.73      0.73      2223
    positive       0.81      0.80      0.81      1717

    accuracy                           0.77      5496
   macro avg       0.77      0.77      0.77      5496
weighted avg       0.77      0.77      0.77      5496


Running: distilbert_u4


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14759.84it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u4 — Trainable params: 28,944,387 / 66,955,779 (43.2%)
  Epoch 1/3 — Loss: 0.6279 — Acc: 0.7357
  Epoch 2/3 — Loss: 0.4946 — Acc: 0.8007
  Epoch 3/3 — Loss: 0.4476 — Acc: 0.8223

distilbert_u4 → F1: 0.7878 | Accuracy: 0.7875
              precision    recall  f1-score   support

    negative       0.78      0.81      0.79      1556
     neutral       0.75      0.76      0.76      2223
    positive       0.85      0.80      0.82      1717

    accuracy                           0.79      5496
   macro avg       0.79      0.79      0.79      5496
weighted avg       0.79      0.79      0.79      5496


Running: distilbert_u6


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10672.26it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u6 — Trainable params: 43,120,131 / 66,955,779 (64.4%)
  Epoch 1/3 — Loss: 0.6028 — Acc: 0.7481
  Epoch 2/3 — Loss: 0.4578 — Acc: 0.8198
  Epoch 3/3 — Loss: 0.3831 — Acc: 0.8553

distilbert_u6 → F1: 0.7915 | Accuracy: 0.7913
              precision    recall  f1-score   support

    negative       0.78      0.82      0.80      1556
     neutral       0.76      0.76      0.76      2223
    positive       0.85      0.81      0.83      1717

    accuracy                           0.79      5496
   macro avg       0.80      0.80      0.80      5496
weighted avg       0.79      0.79      0.79      5496



In [20]:
# Final results: all models including DistilBERT experiments
all_results = {
    "Random Forest": {
        "F1 (weighted)": round(f1_score(y_test, rf_preds,   average='weighted'), 4),
        "Accuracy":      round(accuracy_score(y_test, rf_preds), 4)
    },
    "XGBoost": {
        "F1 (weighted)": round(f1_score(y_test, xgb_preds,  average='weighted'), 4),
        "Accuracy":      round(accuracy_score(y_test, xgb_preds), 4)
    },
    "LSTM": {
        "F1 (weighted)": round(f1_score(y_test, lstm_preds, average='weighted'), 4),
        "Accuracy":      round(accuracy_score(y_test, lstm_preds), 4)
    },
}

# Merge DistilBERT experiment results
all_results.update(experiment_results)

summary = pd.DataFrame(all_results).T
summary = summary.sort_values("F1 (weighted)", ascending=False)
print(summary.to_string())

               F1 (weighted)  Accuracy
distilbert_u6         0.7915    0.7913
distilbert_u4         0.7878    0.7875
distilbert_u2         0.7653    0.7653
XGBoost               0.6972    0.6976
Random Forest         0.6784    0.6801
distilbert_u0         0.5940    0.5966
LSTM                  0.2330    0.4045
